# 🏥 OmniMedical Suite — توليد بيانات تدريب OCR للنصوص اليدوية

دفتر ملاحظات تفاعلي لتوليد بيانات تدريب اصطناعية (Synthetic Data) لنماذج التعرف على النصوص اليدوية (HTR/OCR) العربية والإنجليزية الطبية.

## المزايا:
- **خطوط طبية عربية حقيقية** — تحميل وتثبيت خطوط متنوعة
- **corpus طبي شامل** — وصفات، تقارير مختبر، تشخيصات، أسماء أدوية
- **تعزيزات واقعية** — ضوضاء، تشويه، دوران، انتشار حبر، ضغط JPEG
- **دعم TrOCR** — تجهيز البيانات بصيغة HuggingFace لتدريب TrOCR مع LoRA
- **تنسيقات متعددة** — LMDB, TSV, JSONL, HuggingFace Dataset
- **رؤية مسبقة تفاعلية** — معاينة العينات المولّدة قبل التصدير
- **تصدير إلى Google Drive** — حفظ مباشر لاستخدامه لاحقاً

---

> ⚡ **يُوصى بتشغيله على Google Colab مع مسرّع GPU (T4 أو أفضل)**

In [ ]:
# ══════════════════════════════════════════════════════════════
# 1. تثبيت المكتبات المطلوبة
# ══════════════════════════════════════════════════════════════
!pip install -q --upgrade pip
!pip install -q \
    torch torchvision \
    transformers datasets \
    peft \
    arabic-reshaper python-bidi \
    Pillow opencv-python-headless \
    numpy matplotlib \
    tqdm lmdb \
    albumentations \
    sentencepiece protobuf

# تثبيت محرك PIL لرسم النصوص العربية
!apt-get update -qq
!apt-get install -y -qq fonts-noto-core fonts-noto-extra 2>/dev/null

import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
print('✅ All packages installed successfully')

## 2. إعداد البيئة وتهيئة الأدوات

In [ ]:
# ══════════════════════════════════════════════════════════════
# 2. الإعدادات والاستيرادات
# ══════════════════════════════════════════════════════════════
import os
import re
import json
import random
import hashlib
from pathlib import Path
from typing import Dict, List, Tuple, Optional
from collections import Counter

import numpy as np
import cv2
from PIL import Image, ImageDraw, ImageFont, ImageFilter, ImageEnhance
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
import arabic_reshaper
from bidi.algorithm import get_display

# ── إعداد الخطوط العربية ─────────────────────────────────────
plt.rcParams['font.family'] = 'Noto Sans Arabic'
plt.rcParams['axes.unicode_minus'] = False

# ── مسارات العمل ─────────────────────────────────────────────
BASE_DIR = Path('/content/omni_medical_training')
FONTS_DIR = BASE_DIR / 'fonts'
OUTPUT_DIR = BASE_DIR / 'output'
CORPUS_DIR = BASE_DIR / 'corpus'

BASE_DIR.mkdir(parents=True, exist_ok=True)
FONTS_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CORPUS_DIR.mkdir(parents=True, exist_ok=True)

# ── بذرة عشوائية للتكرارية ─────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

print(f'📂 Base: {BASE_DIR}')
print(f'📂 Fonts: {FONTS_DIR}')
print(f'📂 Output: {OUTPUT_DIR}')
print('✅ Environment ready')

## 3. تحميل خطوط عربية طبية متنوعة

In [ ]:
# ══════════════════════════════════════════════════════════════
# 3. تحميل وتجهيز الخطوط العربية
# ══════════════════════════════════════════════════════════════

# ──搜集系统中 الخطوط العربية المتوفرة ───────────────────────
def find_system_fonts() -> List[Path]:
    """البحث عن خطوط عربية في النظام."""
    font_dirs = [
        Path('/usr/share/fonts'),
        Path('/usr/local/share/fonts'),
    ]
    extensions = ['.ttf', '.otf', '.ttc']
    fonts = []
    for d in font_dirs:
        if d.exists():
            for ext in extensions:
                fonts.extend(d.rglob(f'*{ext}'))
    return sorted(fonts)

system_fonts = find_system_fonts()
print(f'🔍 Found {len(system_fonts)} system fonts')

# ── تحميل خطوط عربية إضافية من Google Fonts ─────────────────
!cd {FONTS_DIR} && \\
wget -q 'https://github.com/google/fonts/raw/main/ofl/amiri/Amiri-Regular.ttf' -O Amiri-Regular.ttf 2>/dev/null; \\
wget -q 'https://github.com/google/fonts/raw/main/ofl/cairo/Cairo%5Bslnt%2Cwght%5D.ttf' -O Cairo-Variable.ttf 2>/dev/null; \\
wget -q 'https://github.com/google/fonts/raw/main/ofl/notosansarabic/NotoSansArabic%5Bwdth%2Cwght%5D.ttf' -O NotoSansArabic-Variable.ttf 2>/dev/null; \\
wget -q 'https://github.com/google/fonts/raw/main/ofl/notoserifarabic/NotoSerifArabic%5Bwdth%2Cwght%5D.ttf' -O NotoSerifArabic-Variable.ttf 2>/dev/null; \\
wget -q 'https://github.com/google/fonts/raw/main/ofl/marhey/Marhey%5Bwght%5D.ttf' -O Marhey-Variable.ttf 2>/dev/null; \\
wget -q 'https://github.com/google/fonts/raw/main/ofl/lateef/Lateef-Regular.ttf' -O Lateef-Regular.ttf 2>/dev/null; \\
wget -q 'https://github.com/google/fonts/raw/main/ofl/scheherazade/ScheherazadeNew-Regular.ttf' -O ScheherazadeNew-Regular.ttf 2>/dev/null; \\
wget -q 'https://github.com/aliftype/janna/raw/main/Janna-Regular.ttf' -O Janna-Regular.ttf 2>/dev/null; \\
wget -q 'https://github.com/google/fonts/raw/main/ofl/tajawal/Tajawal-Regular.ttf' -O Tajawal-Regular.ttf 2>/dev/null; \\
wget -q 'https://github.com/google/fonts/raw/main/ofl/changa/Changa%5Bwght%5D.ttf' -O Changa-Variable.ttf 2>/dev/null; \\
wget -q 'https://github.com/aliftype/kufi-arabic/raw/main/KufiArabic-Regular.ttf' -O KufiArabic-Regular.ttf 2>/dev/null; \\
wget -q 'https://github.com/google/fonts/raw/main/ofl/reemkufi/ReemKufi%5Bwght%5D.ttf' -O ReemKufi-Variable.ttf 2>/dev/null; \\
echo 'Download complete'

# ── جمع جميع الخطوط المتوفرة ──────────────────────────────────
downloaded_fonts = list(FONTS_DIR.glob('*.ttf')) + list(FONTS_DIR.glob('*.otf'))
all_fonts = downloaded_fonts + system_fonts
valid_fonts = []
for f in all_fonts:
    try:
        _ = ImageFont.truetype(str(f), 20)
        valid_fonts.append(f)
    except Exception:
        pass

print(f'\n✅ {len(valid_fonts)} valid fonts loaded')
for f in valid_fonts[:5]:
    print(f'   📝 {f.name}')
if len(valid_fonts) > 5:
    print(f'   ... and {len(valid_fonts) - 5} more')

## 4. بناء Corpus طبي عربي/إنجليزي شامل

In [ ]:
# ══════════════════════════════════════════════════════════════
# 4. بناء Corpus طبي شامل (عربي + إنجليزي)
# ══════════════════════════════════════════════════════════════

MEDICAL_CORPUS = {
    # ── وصفات طبية عربية ──────────────────────────────────
    'prescriptions_ar': [
        'قرص باراسيتامول 500 ملغ مرتين يوميا بعد الأكل',
        'كبسولة أموكسيسيلين 250 ملغ ثلاث مرات يوميا لمدة 7 أيام',
        'شراب ibuprofen 5 مل كل 8 ساعات حسب الوزن',
        'حقنة ميتوبريبروليد 1 مل عضل مرة واحدة',
        'قطرة توباميد عينين 4 مرات يوميا',
        'مرهم ديكلوفيك 1% موضعيا مرتين يوميا',
        'قرص أملوديبين 5 ملغ صباحا لمدة 30 يوم',
        'كبسولة أزيثرومايسين 500 ملغ مرة واحدة يوميا لمدة 3 أيام',
        'حقنة إنسولين 10 وحدات تحت الجلد قبل الأكل',
        'قرص ميثوتريكسات 7.5 ملغ أسبوعيا يوم الجمعة',
        'شراب سيتيريزين 5 مل مرة واحدة يوميا',
        'كبسولة أوميبرازول 20 ملغ قبل الفطور لمدة 14 يوم',
        'قرص كلوبيدوغريل 75 ملغ مرة واحدة يوميا',
        'شراب سالبوتامول 2 مل ثلاث مرات يوميا',
        'قرص ليفتيروكسين 50 ميكروغرام صباحا على معدة فارغة',
        'حقنة لورازيبام 2 ملغ عضل عند الحاجة',
        'كبسولة دوكسيسيكلين 100 ملغ مرتين يوميا لمدة 14 يوم',
        'مرهم ميترونيدازول موضعيا مرتين يوميا لمدة 10 أيام',
        'قرص لوسارتان 50 ملغ مرة واحدة يوميا',
        'كبسولة إيتوريكوكسيلب 60 ملغ مرة واحدة يوميا',
    ],

    # ── تقارير مختبر عربية ─────────────────────────────────
    'lab_reports_ar': [
        'تحليل الدم الشامل: الهيموغلوبين 12.5 جرام/ديسيلتر',
        'الكرات البيضاء 7500 خلية/ميكرولتر',
        'الصفيحات الدموية 250000 خلية/ميكرولتر',
        'معدل ترسب الكريات الحمراء 25 ملم/ساعة',
        'بروتين سي التفاعلي 8 ملغ/لتر',
        'سكر الدم الصائم 110 ملغ/ديسيلتر',
        'الهيموغلوبين السكري 7.2 بالمئة',
        'الكرياتينين في المصل 0.9 ملغ/ديسيلتر',
        'اليوريا في الدم 20 ملغ/ديسيلتر',
        'إنزيمات الكبد ناقلة الأمين 35 وحدة/لتر',
        'الفوسفاتاز القلوية 120 وحدة/لتر',
        'البروتين الكلي 7.2 جرام/ديسيلتر',
        'الألبومين 4.1 جرام/ديسيلتر',
        'البيليروبين الكلي 0.8 ملغ/ديسيلتر',
        'الزمن الجزئي للثرومبوبلاستين 28 ثانية',
        'معدل التروبوبلاستين 12 ثانية',
        'إنزيم كيناز الكرياتين 200 وحدة/لتر',
        'نسبة تطبيع دولية 1.1',
        'حمض اليوريك 5.2 ملغ/ديسيلتر',
    ],

    # ── تشخيصات عربية ──────────────────────────────────────
    'diagnoses_ar': [
        'التشخيص: ارتفاع ضغط الدم المزمن',
        'داء السكري النوع الثاني',
        'التهاب المفاصل الروماتويدي',
        'خشونة المفاصل في الركبة اليمنى',
        'كسر كوليس في المعصم الأيسر',
        'التهاب الجيوب الأنفية المزمن',
        'التهاب الشعب الهوائية الحاد',
        'التهاب المعدة والمريء',
        'حصوات المرارة',
        'قصور كلوي مزمن المرحلة الثالثة',
        'كسر في عنق الفخذ الأيسر',
        'انزلاق غضروفي قطني',
        'ورم الخلية العملاقة في الطرف القريب لعظم الظنبوب',
        'ساركوما عظمية في distal femur',
        'داء باجيت في العظم الحرقفي',
        'نخر لاوعائي في رأس الفخذ',
        'التهاب العظم والنخاع المزمن',
        'خلل تطور الورك التنموي',
        'التهاب الفقار المقسط',
    ],

    # ── معلومات المرضى ──────────────────────────────────────
    'patient_info_ar': [
        'اسم المريض: أحمد محمد العلي',
        'رقم الملف: 45231',
        'تاريخ الولادة: 1985/03/15',
        'الفصيلة: A+',
        'الجنس: ذكر',
        'اسم المريض: فاطمة خالد المنصور',
        'رقم الملف: 78912',
        'تاريخ الولادة: 1992/07/22',
        'الفصيلة: O-',
        'الجنس: أنثى',
        'الطبيب المعالج: د. عبدالله المنصور',
        'التخصص: جراحة العظام',
        'التاريخ: 2026/05/27',
        'اسم المريض: نورة سعد القحطاني',
        'العمر: 5 سنوات',
        'الوزن: 18 كيلوجرام',
    ],

    # ── English medical terms ────────────────────────────────
    'medical_en': [
        'Patient: John Smith',
        'Date of Birth: 15/03/1985',
        'Blood Type: A Positive',
        'Diagnosis: Osteoarthritis of the right knee',
        'Colles Fracture of the left wrist',
        'Non-displaced femoral neck fracture',
        'Giant Cell Tumor of distal femur',
        'Osteosarcoma of the distal femur',
        'Osteomyelitis of the tibia',
        'Paget Disease of the pelvis',
        'Avascular Necrosis of the femoral head',
        'Rheumatoid Arthritis',
        'Ankylosing Spondylitis',
        'Developmental Dysplasia of the Hip',
        'Prescription: Paracetamol 500mg twice daily',
        'Amoxicillin 250mg three times daily for 7 days',
        'Metformin 500mg twice daily with meals',
        'Aspirin 100mg once daily',
        'WBC count: 7500 cells/microL',
        'Hemoglobin: 12.5 g/dL',
        'Platelet count: 250000/microL',
        'ESR: 25 mm/hr',
        'CRP: 8 mg/L',
        'Creatinine: 0.9 mg/dL',
        'Fasting glucose: 110 mg/dL',
        'HbA1c: 7.2%',
    ],

    # ── أرقام وقيم طبية ──────────────────────────────────────
    'numbers_medical': [
        '120/80 ملم زئبق',
        '72 نبضة في الدقيقة',
        '18 تنفس في الدقيقة',
        '36.8 درجة مئوية',
        '98 بالمئة تشبع الأكسجين',
        '75 كيلوجرام',
        '170 سنتيمتر',
        '25.9 مؤشر كتلة الجسم',
        '90/60 ملم زئبق',
        '110 نبضة في الدقيقة',
        '38.5 درجة مئوية',
        '92 بالمئة تشبع الأكسجين',
        '22 تنفس في الدقيقة',
    ],
    # ── تقرير أشعة ──────────────────────────────────────────
    'radiology_ar': [
        'صورة شعاعية للصدر الأمامية الخلفية',
        'لا يوجد ارتشاح رئوي',
        'القلب بحجم طبيعي',
        'صورة شعاعية للمعصم الجانبية',
        'كسر كوليس مع انزياح خلفي',
        'تصوير بالرنين المغناطيسي للركبة',
        'تمزق في الرباط الصليبي الأمامي',
        'صورة مقطعية للبطن مع وسيط',
        'لا يوجد كتلة مجسوسة',
        'الكبد والطحال بحجم طبيعي',
        'المسح العظمي يظهر نقائل عظمية متعددة',
        'صورة شعاعية للعمود الفقري القطني',
        'انزلاق غضروفي بين الفقرة L4 و L5',
    ],
}

# ── دمج جميع النصوص ──────────────────────────────────────────
all_texts = []
for category, texts in MEDICAL_CORPUS.items():
    all_texts.extend(texts)

print(f'📚 Total corpus size: {len(all_texts)} entries')
for cat, texts in MEDICAL_CORPUS.items():
    print(f'   📋 {cat}: {len(texts)} entries')

# ── حفظ الـ corpus ──────────────────────────────────────────────
corpus_file = CORPUS_DIR / 'medical_corpus.txt'
with open(corpus_file, 'w', encoding='utf-8') as f:
    for text in all_texts:
        f.write(text + '\n')
print(f'\n💾 Corpus saved to {corpus_file}')

## 5. محرك توليد البيانات الاصطناعية

In [ ]:
# ══════════════════════════════════════════════════════════════
# 5. محرك توليد الصور الاصطناعية
# ══════════════════════════════════════════════════════════════

class MedicalTextRenderer:
    """رسم النصوص الطبية العربية والإنجليزية على صور."""

    def __init__(self, fonts: List[Path]):
        self.fonts = fonts
        # ألوان الحبر المتنوعة (تقليد أقلام حقيقية)
        self.ink_colors = [
            (0, 0, 0),         # أسود
            (15, 15, 15),       # رمادي داكن جداً
            (30, 30, 60),       # أزرق داكن (قلم أزرق)
            (50, 20, 20),       # بني غامق (قلم بني)
            (0, 0, 120),        # أزرق (حبر أزرق طبي)
            (100, 0, 0),        # أحمر داكن (طابعة نقية)
        ]

    def is_arabic(self, text: str) -> bool:
        """فحص ما إذا كان النص يحتوي أحرف عربية."""
        arabic_range = range(0x0600, 0x06FF + 1)
        return any(ord(c) in arabic_range for c in text)

    def reshape_arabic(self, text: str) -> str:
        """إعادة تشكيل النص العربي للعرض الصحيح."""
        try:
            reshaped = arabic_reshaper.reshape(text)
            bidi_text = get_display(reshaped)
            return bidi_text
        except Exception:
            return text

    def render(self, text: str, font: ImageFont.FreeTypeFont = None,
                font_size: int = 32) -> Image.Image:
        """رسم نص على صورة بيضاء."""
        # اختيار خط عشوائي إن لم يُحدد
        if font is None:
            font_path = random.choice(self.fonts)
            try:
                font = ImageFont.truetype(str(font_path), font_size)
            except Exception:
                font = ImageFont.load_default()

        # إعادة تشكيل العربي
        if self.is_arabic(text):
            text = self.reshape_arabic(text)

        # حساب حجم النص
        dummy = Image.new('RGB', (1, 1))
        draw = ImageDraw.Draw(dummy)
        try:
            bbox = draw.textbbox((0, 0), text, font=font)
            text_w = bbox[2] - bbox[0]
            text_h = bbox[3] - bbox[1]
        except Exception:
            text_w = len(text) * font_size // 2
            text_h = font_size

        # إنشاء الصورة مع هوامش
        padding_x = random.randint(30, 80)
        padding_y = random.randint(20, 50)
        w = text_w + padding_x * 2
        h = text_h + padding_y * 2

        img = Image.new('RGB', (w, h), (255, 255, 255))
        draw = ImageDraw.Draw(img)

        # لون الحبر
        ink = random.choice(self.ink_colors)

        # رسم النص
        x = padding_x
        y = (h - text_h) // 2
        draw.text((x, y), text, font=font, fill=ink)

        return img


class MedicalAugmenter:
    """تعزيزات واقعية تحاكي الكتابة اليدوية والمستندات الطبية الممسوحة."""

    def __init__(self, intensity: str = 'medium'):
        """
        Args:
            intensity: مستوى التعزيز ('easy', 'medium', 'hard')
        """
        self.intensity = intensity
        self.intensity_map = {
            'easy': 0.3,
            'medium': 0.6,
            'hard': 1.0,
        }

    def apply(self, image: Image.Image) -> Image.Image:
        """تطبيق تعزيزات عشوائية."""
        prob = self.intensity_map.get(self.intensity, 0.6)
        img = image.copy()

        # اختيار 2-6 تعزيزات عشوائية
        num_augs = random.randint(2, 6)
        augmentations = random.sample([
            self._add_gaussian_noise,
            self._add_salt_pepper_noise,
            self._add_motion_blur,
            self._add_gaussian_blur,
            self._adjust_contrast,
            self._adjust_brightness,
            self._add_rotation,
            self._add_perspective,
            self._add_ink_bleed,
            self._add_paper_texture,
            self._add_compression,
            self._add_shadow,
            self._add_erasing,
            self._add_elastic_distortion,
        ], num_augs)

        for aug_fn in augmentations:
            if random.random() < prob:
                img = aug_fn(img)

        return img

    def _add_gaussian_noise(self, img: Image.Image) -> Image.Image:
        arr = np.array(img).astype(np.int16)
        noise = np.random.normal(0, random.randint(8, 25), arr.shape)
        noisy = np.clip(arr + noise, 0, 255).astype(np.uint8)
        return Image.fromarray(noisy)

    def _add_salt_pepper_noise(self, img: Image.Image) -> Image.Image:
        arr = np.array(img)
        prob = random.uniform(0.001, 0.01)
        salt = np.random.random(arr.shape[:2]) < prob / 2
        pepper = np.random.random(arr.shape[:2]) < prob / 2
        arr[salt] = 255
        arr[pepper] = 0
        return Image.fromarray(arr)

    def _add_motion_blur(self, img: Image.Image) -> Image.Image:
        arr = np.array(img)
        size = random.randint(3, 10)
        kernel = np.zeros((size, size))
        kernel[size // 2, :] = 1.0 / size
        blurred = cv2.filter2D(arr, -1, kernel)
        return Image.fromarray(blurred)

    def _add_gaussian_blur(self, img: Image.Image) -> Image.Image:
        radius = random.uniform(0.3, 1.5)
        return img.filter(ImageFilter.GaussianBlur(radius))

    def _adjust_contrast(self, img: Image.Image) -> Image.Image:
        factor = random.uniform(0.6, 1.4)
        return ImageEnhance.Contrast(img).enhance(factor)

    def _adjust_brightness(self, img: Image.Image) -> Image.Image:
        factor = random.uniform(0.7, 1.3)
        return ImageEnhance.Brightness(img).enhance(factor)

    def _add_rotation(self, img: Image.Image) -> Image.Image:
        angle = random.uniform(-5, 5)
        return img.rotate(angle, fillcolor=(255, 255, 255), expand=True)

    def _add_perspective(self, img: Image.Image) -> Image.Image:
        arr = np.array(img)
        h, w = arr.shape[:2]
        pts1 = np.float32([[0, 0], [w, 0], [0, h], [w, h]])
        offset = random.randint(0, max(1, w // 15))
        pts2 = np.float32([
            [offset, offset],
            [w - offset, random.randint(0, offset)],
            [random.randint(0, offset), h - offset],
            [w - offset, h - offset],
        ])
        M = cv2.getPerspectiveTransform(pts1, pts2)
        warped = cv2.warpPerspective(arr, M, (w, h),
                                       borderMode=cv2.BORDER_REPLICATE)
        return Image.fromarray(warped)

    def _add_ink_bleed(self, img: Image.Image) -> Image.Image:
        arr = np.array(img)
        if len(arr.shape) == 3:
            gray = cv2.cvtColor(arr, cv2.COLOR_RGB2GRAY)
        else:
            gray = arr.copy()
        kernel = np.ones((2, 2), np.uint8)
        dilated = cv2.dilate(255 - gray, kernel, iterations=random.randint(1, 2))
        result = 255 - (dilated * random.uniform(0.3, 0.7)).astype(np.uint8)
        if len(arr.shape) == 3:
            return Image.fromarray(cv2.cvtColor(result, cv2.COLOR_GRAY2RGB))
        return Image.fromarray(result)

    def _add_paper_texture(self, img: Image.Image) -> Image.Image:
        arr = np.array(img).astype(np.float32)
        h, w = arr.shape[:2]
        texture = np.random.normal(245, 8, (h, w, 3)).astype(np.float32)
        alpha = random.uniform(0.85, 0.98)
        blended = arr * alpha + texture * (1 - alpha)
        return Image.fromarray(np.clip(blended, 0, 255).astype(np.uint8))

    def _add_compression(self, img: Image.Image) -> Image.Image:
        import io
        quality = random.randint(40, 85)
        buffer = io.BytesIO()
        img.save(buffer, format='JPEG', quality=quality)
        buffer.seek(0)
        return Image.open(buffer).convert('RGB')

    def _add_shadow(self, img: Image.Image) -> Image.Image:
        arr = np.array(img).astype(np.float32)
        h, w = arr.shape[:2]
        # ظل متدرج من الحافة
        for i in range(min(30, w // 4)):
            factor = 1.0 - (i / 30) * random.uniform(0.1, 0.3)
            arr[:, i, :] = np.clip(arr[:, i, :] * factor, 0, 255)
        return Image.fromarray(arr.astype(np.uint8))

    def _add_erasing(self, img: Image.Image) -> Image.Image:
        arr = np.array(img)
        h, w = arr.shape[:2]
        # محو جزء عشوائي صغير
        num_erasures = random.randint(0, 3)
        for _ in range(num_erasures):
            ex = random.randint(0, max(1, w - 20))
            ey = random.randint(0, max(1, h - 5))
            ew = random.randint(10, min(50, w - ex))
            eh = random.randint(2, min(8, h - ey))
            arr[ey:ey + eh, ex:ex + ew] = 255
        return Image.fromarray(arr)

    def _add_elastic_distortion(self, img: Image.Image) -> Image.Image:
        arr = np.array(img)
        h, w = arr.shape[:2]
        dx = np.random.uniform(-3, 3, (h, w)).astype(np.float32)
        dy = np.random.uniform(-3, 3, (h, w)).astype(np.float32)
        x, y = np.meshgrid(np.arange(w), np.arange(h))
        x_new = np.clip(x + dx, 0, w - 1).astype(np.int32)
        y_new = np.clip(y + dy, 0, h - 1).astype(np.int32)
        if len(arr.shape) == 3:
            distorted = arr[y_new, x_new]
        else:
            distorted = arr[y_new, x_new]
        return Image.fromarray(distorted)


class SyntheticDatasetGenerator:
    """مولد مجموعات البيانات الاصطناعية الطبية."""

    def __init__(self, corpus: List[str], fonts: List[Path]):
        self.corpus = corpus
        self.renderer = MedicalTextRenderer(fonts)
        self.augmenter = MedicalAugmenter(intensity='medium')

    def generate_sample(self, text: str = None,
                          difficulty: str = 'medium',
                          apply_augmentation: bool = True) -> Tuple[Image.Image, str]:
        """
        توليد عينة واحدة.

        Args:
            text: النص (اختياري - يتم اختياره عشوائياً من corpus)
            difficulty: 'easy', 'medium', 'hard'
            apply_augmentation: تطبيق التعزيزات

        Returns:
            (صورة, النص)
        """
        if text is None:
            text = random.choice(self.corpus)

        # حجم الخط حسب الصعوبة
        size_map = {'easy': (28, 40), 'medium': (22, 36), 'hard': (18, 30)}
        size_range = size_map.get(difficulty, (22, 36))
        font_size = random.randint(*size_range)

        # رسم النص
        image = self.renderer.render(text, font_size=font_size)

        # تعزيزات
        if apply_augmentation:
            self.augmenter.intensity = difficulty
            image = self.augmenter.apply(image)

        return image, text

    def generate_batch(self, num_samples: int,
                        split: str = 'train',
                        difficulty_dist: Dict[str, float] = None,
                        save_images: bool = True) -> List[Dict]:
        """
        توليد دفعة كاملة.

        Args:
            num_samples: عدد العينات
            split: 'train', 'val', 'test'
            difficulty_dist: توزيع الصعوبة {'easy': 0.3, 'medium': 0.5, 'hard': 0.2}
            save_images: حفظ الصور على القرص

        Returns:
            قائمة بالبيانات الوصفية
        """
        if difficulty_dist is None:
            difficulty_dist = {'easy': 0.3, 'medium': 0.5, 'hard': 0.2}

        split_dir = OUTPUT_DIR / split
        images_dir = split_dir / 'images'
        images_dir.mkdir(parents=True, exist_ok=True)

        metadata = []

        for i in tqdm(range(num_samples), desc=f'🎨 Generating {split}'):
            # اختيار الصعوبة
            r = random.random()
            cumulative = 0.0
            difficulty = 'medium'
            for diff, prob in difficulty_dist.items():
                cumulative += prob
                if r <= cumulative:
                    difficulty = diff
                    break

            # توليد العينة
            image, text = self.generate_sample(difficulty=difficulty)

            # حفظ
            filename = f'{split}_{i:06d}.png'
            if save_images:
                image.save(images_dir / filename, 'PNG')

            metadata.append({
                'filename': filename,
                'text': text,
                'difficulty': difficulty,
                'width': image.width,
                'height': image.height,
            })

        # حفظ التسميات
        labels_file = split_dir / 'labels.txt'
        with open(labels_file, 'w', encoding='utf-8') as f:
            f.write('filename\ttext\tdifficulty\n')
            for m in metadata:
                f.write(f"{m['filename']}\t{m['text']}\t{m['difficulty']}\n")

        # حفظ الـ metadata
        meta_file = split_dir / 'metadata.json'
        with open(meta_file, 'w', encoding='utf-8') as f:
            json.dump(metadata, f, ensure_ascii=False, indent=2)

        print(f'✅ {split}: {len(metadata)} samples -> {split_dir}')
        return metadata


# ── إنشاء المولد ──────────────────────────────────────────────
generator = SyntheticDatasetGenerator(all_texts, valid_fonts)
print('✅ Synthetic Data Generator ready')
print(f'   Fonts: {len(valid_fonts)}')
print(f'   Corpus: {len(all_texts)} texts')
print(f'   Augmentations: 14 types')
print(f'   Difficulty levels: easy / medium / hard')

## 6. توليد العينات ومعاينتها

In [ ]:
# ══════════════════════════════════════════════════════════════
# 6. توليد عينات تجريبية ومعاينتها
# ══════════════════════════════════════════════════════════════

# توليد 12 عينة تجريبية بثلاث مستويات صعوبة
preview_samples = []
difficulties = ['easy', 'medium', 'hard']

fig, axes = plt.subplots(4, 3, figsize=(18, 16))
fig.suptitle('نماذج من بيانات التدريب المولّدة — Medical OCR', fontsize=16)

for row, difficulty in enumerate(difficulties):
    for col in range(4):
        img, text = generator.generate_sample(difficulty=difficulty)
        preview_samples.append((img, text, difficulty))

        ax = axes[row, col] if len(difficulties) > 1 else axes[col]
        ax.imshow(np.array(img))
        ax.set_title(f'[{difficulty}] {text[:40]}...', fontsize=9)
        ax.axis('off')

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'preview_samples.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'🖼️ Preview saved to {OUTPUT_DIR / "preview_samples.png"}')

## 7. توليد مجموعة البيانات الكاملة

In [ ]:
# ══════════════════════════════════════════════════════════════
# 7. توليد مجموعة البيانات الكاملة
# ══════════════════════════════════════════════════════════════

# ── إعدادات التوليد ───────────────────────────────────────────
NUM_TRAIN = 8000     # 8000 عينة تدريب
NUM_VAL = 1000       # 1000 عينة تحقق
NUM_TEST = 1000      # 1000 عينة اختبار

print(f'🚀 Starting dataset generation...')
print(f'   Train: {NUM_TRAIN} | Val: {NUM_VAL} | Test: {NUM_TEST}')
print(f'   Total: {NUM_TRAIN + NUM_VAL + NUM_TEST} samples')
print()

# ── توليد التقسيمات ──────────────────────────────────────────
train_meta = generator.generate_batch(NUM_TRAIN, split='train')
val_meta = generator.generate_batch(NUM_VAL, split='val')
test_meta = generator.generate_batch(NUM_TEST, split='test')

print(f'\n📊 Summary:')
print(f'   Train: {len(train_meta)} samples')
print(f'   Val: {len(val_meta)} samples')
print(f'   Test: {len(test_meta)} samples')
print(f'   Total: {len(train_meta) + len(val_meta) + len(test_meta)} samples')

# ── حفظ إعدادات التوليد ────────────────────────────────────────
gen_config = {
    'num_train': NUM_TRAIN,
    'num_val': NUM_VAL,
    'num_test': NUM_TEST,
    'total': NUM_TRAIN + NUM_VAL + NUM_TEST,
    'num_fonts': len(valid_fonts),
    'corpus_size': len(all_texts),
    'seed': SEED,
    'augmentations': 14,
    'difficulty_distribution': {'easy': 0.3, 'medium': 0.5, 'hard': 0.2},
}

with open(OUTPUT_DIR / 'generation_config.json', 'w', encoding='utf-8') as f:
    json.dump(gen_config, f, indent=2, ensure_ascii=False)

print(f'\n💾 Config saved to {OUTPUT_DIR / "generation_config.json"}')

## 8. تصدير البيانات بصيغ متعددة

In [ ]:
# ══════════════════════════════════════════════════════════════
# 8. تصدير البيانات بصيغ متعددة
# ══════════════════════════════════════════════════════════════

import shutil
import pickle

def export_to_lmdb(metadata: List[Dict], split: str) -> Path:
    """تصدير بتنسيق LMDB (سريع للتدريب)."""
    lmdb_path = OUTPUT_DIR / f'{split}.lmdb'
    if lmdb_path.exists():
        shutil.rmtree(lmdb_path)

    env = lmdb.open(str(lmdb_path), map_size=1099511627776)  # 1TB
    images_dir = OUTPUT_DIR / split / 'images'

    with env.begin(write=True) as txn:
        for idx, sample in enumerate(tqdm(metadata, desc=f'💾 LMDB {split}')):
            img_path = images_dir / sample['filename']
            with open(img_path, 'rb') as f:
                image_bytes = f.read()
            value = pickle.dumps({
                'image': image_bytes,
                'text': sample['text'],
                'difficulty': sample['difficulty'],
            })
            txn.put(f'{idx:08d}'.encode(), value)
        txn.put(b'__len__', str(len(metadata)).encode())

    env.close()
    print(f'✅ LMDB {split}: {len(metadata)} samples -> {lmdb_path}')
    return lmdb_path


def export_to_hf_dataset(metadata: List[Dict], split: str) -> Path:
    """تصدير بتنسيق HuggingFace Dataset."""
    from datasets import Dataset, Features, Value, Image as HFImage

    images_dir = OUTPUT_DIR / split / 'images'
    images, texts, difficulties = [], [], []

    for sample in tqdm(metadata, desc=f'🤗 HF {split}'):
        img = Image.open(images_dir / sample['filename']).convert('RGB')
        images.append(img)
        texts.append(sample['text'])
        difficulties.append(sample['difficulty'])

    hf_dataset = Dataset.from_dict({
        'image': images,
        'text': texts,
        'difficulty': difficulties,
    })

    hf_path = OUTPUT_DIR / f'{split}_hf'
    hf_dataset.save_to_disk(str(hf_path))
    print(f'✅ HF Dataset {split}: {len(metadata)} samples -> {hf_path}')
    return hf_path


def export_to_jsonl(metadata: List[Dict], split: str) -> Path:
    """تصدير بتنسيق JSONL."""
    images_dir = OUTPUT_DIR / split / 'images'
    jsonl_path = OUTPUT_DIR / f'{split}.jsonl'

    with open(jsonl_path, 'w', encoding='utf-8') as f:
        for sample in metadata:
            entry = {
                'image_path': str(images_dir / sample['filename']),
                'text': sample['text'],
                'difficulty': sample['difficulty'],
            }
            f.write(json.dumps(entry, ensure_ascii=False) + '\n')

    print(f'✅ JSONL {split}: {len(metadata)} samples -> {jsonl_path}')
    return jsonl_path


# ── تصدير جميع التقسيمات ─────────────────────────────────────
print('📤 Exporting datasets in multiple formats...\n')

# LMDB
export_to_lmdb(train_meta, 'train')
export_to_lmdb(val_meta, 'val')
export_to_lmdb(test_meta, 'test')

print()

# HuggingFace
export_to_hf_dataset(train_meta, 'train')
export_to_hf_dataset(val_meta, 'val')
export_to_hf_dataset(test_meta, 'test')

print()

# JSONL
export_to_jsonl(train_meta, 'train')
export_to_jsonl(val_meta, 'val')
export_to_jsonl(test_meta, 'test')

print('\n✅ All formats exported successfully!')

## 9. تصدير إلى Google Drive

In [ ]:
# ══════════════════════════════════════════════════════════════
# 9. تصدير مجموعة البيانات إلى Google Drive
# ══════════════════════════════════════════════════════════════

from google.colab import drive

DRIVE_DIR = Path('/content/drive/MyDrive/omni_medical_training')

try:
    drive.mount('/content/drive')
    DRIVE_DIR.mkdir(parents=True, exist_ok=True)

    # نسخ جميع الملفات
    import shutil
    
    # إنشاء أرشيف ZIP
    archive_path = BASE_DIR.parent / 'omni_medical_training.zip'
    !cd {BASE_DIR.parent} && zip -r -q {archive_path} omni_medical_training/

    # نسخ إلى Drive
    shutil.copy2(archive_path, DRIVE_DIR / 'omni_medical_training.zip')

    # حساب الحجم
    size_mb = archive_path.stat().st_size / (1024 * 1024)
    print(f'\n✅ Dataset exported to Google Drive!')
    print(f'   📁 {DRIVE_DIR / "omni_medical_training.zip"}')
    print(f'   📦 Size: {size_mb:.1f} MB')
    
    # عرض محتويات المجلد
    print(f'\n📂 Dataset contents:')
    for p in sorted(BASE_DIR.rglob('*')):
        if p.is_file():
            rel = p.relative_to(BASE_DIR)
            size_kb = p.stat().st_size / 1024
            print(f'   {rel} ({size_kb:.1f} KB)')

except Exception as e:
    print(f'⚠️ Google Drive not available: {e}')
    print('💾 Dataset is ready in Colab at:', BASE_DIR)
    print('   Download manually from the file browser on the left.')

## 10. تجهيز البيانات لتدريب TrOCR + LoRA

In [ ]:
# ══════════════════════════════════════════════════════════════
# 10. تجهيز البيانات لتدريب TrOCR مع LoRA
# ══════════════════════════════════════════════════════════════

from transformers import TrOCRProcessor, VisionEncoderDecoderModel
from torch.utils.data import Dataset, DataLoader

# ── تحميل المعالج والنموذج ──────────────────────────────────
MODEL_NAME = 'microsoft/trocr-large-handwritten'

print(f'📥 Loading TrOCR processor and model: {MODEL_NAME}')
processor = TrOCRProcessor.from_pretrained(MODEL_NAME)
model = VisionEncoderDecoderModel.from_pretrained(MODEL_NAME)

# ── توسيع المفردات بالأحرف العربية ────────────────────────────
arabic_chars = 'ابتثجحخدذرزسشصضطظعغفقكلمنهويءآأإةؤى'
vocab = processor.tokenizer.get_vocab()
new_tokens = [c for c in arabic_chars if c not in vocab]

if new_tokens:
    processor.tokenizer.add_tokens(new_tokens)
    model.decoder.resize_token_embeddings(len(processor.tokenizer))
    print(f'➕ Added {len(new_tokens)} Arabic tokens to vocabulary')

# ── فئة مجموعة البيانات ───────────────────────────────────────
class HTRDataset(Dataset):
    """مجموعة بيانات HTR لتدريب TrOCR."""

    def __init__(self, metadata: List[Dict], processor, split: str = 'train',
                 max_target_length: int = 128):
        self.processor = processor
        self.max_target_length = max_target_length
        self.split = split
        self.samples = []

        images_dir = OUTPUT_DIR / split / 'images'
        for m in metadata:
            img_path = images_dir / m['filename']
            if img_path.exists():
                self.samples.append({
                    'image_path': str(img_path),
                    'text': m['text'],
                })

        print(f'📊 {split}: {len(self.samples)} valid samples loaded')

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        sample = self.samples[idx]
        image = Image.open(sample['image_path']).convert('RGB')

        pixel_values = self.processor(
            image, return_tensors='pt'
        ).pixel_values.squeeze()

        labels = self.processor.tokenizer(
            sample['text'],
            padding='max_length',
            max_length=self.max_target_length,
            truncation=True,
            return_tensors='pt'
        ).input_ids.squeeze()

        labels[labels == self.processor.tokenizer.pad_token_id] = -100

        return {'pixel_values': pixel_values, 'labels': labels}


# ── إنشاء مجموعات البيانات ────────────────────────────────────
train_dataset = HTRDataset(train_meta, processor, 'train')
val_dataset = HTRDataset(val_meta, processor, 'val')

# ── DataLoaders ────────────────────────────────────────────────
train_loader = DataLoader(
    train_dataset, batch_size=4, shuffle=True, num_workers=2, pin_memory=True
)
val_loader = DataLoader(
    val_dataset, batch_size=4, shuffle=False, num_workers=2, pin_memory=True
)

print(f'\n✅ TrOCR datasets ready!')
print(f'   Train batches: {len(train_loader)}')
print(f'   Val batches: {len(val_loader)}')
print(f'\n🚀 Ready for training with LoRA!')

## 11. اختبار سريع: تشغيل TrOCR على عينات مولّدة

In [ ]:
# ══════════════════════════════════════════════════════════════
# 11. اختبار سريع: تشغيل TrOCR على العينات المولّدة
# ══════════════════════════════════════════════════════════════

import torch

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)
model.eval()

# اختبار على 6 عينات عشوائية
test_samples = random.sample(test_meta, min(6, len(test_meta)))
test_dir = OUTPUT_DIR / 'test' / 'images'

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle('TrOCR Inference on Synthetic Medical Data', fontsize=14)

with torch.no_grad():
    for idx, sample in enumerate(test_samples):
        img = Image.open(test_dir / sample['filename']).convert('RGB')
        gt_text = sample['text']

        # Run TrOCR
        pixel_values = processor(img, return_tensors='pt').pixel_values.to(device)
        generated_ids = model.generate(pixel_values)
        predicted_text = processor.batch_decode(
            generated_ids, skip_special_tokens=True
        )[0]

        ax = axes[idx // 3, idx % 3]
        ax.imshow(img)
        ax.set_title(
            f'GT: {gt_text[:35]}...\nPred: {predicted_text[:35]}...',
            fontsize=8
        )
        ax.axis('off')

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'trocr_preview.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'🖼️ TrOCR preview saved to {OUTPUT_DIR / "trocr_preview.png"}')

## 12. ملخص ومعلومات مجموعة البيانات

In [ ]:
# ══════════════════════════════════════════════════════════════
# 12. ملخص نهائي
# ══════════════════════════════════════════════════════════════

print('=' * 60)
print('  OmniMedical Suite — Training Data Generation Complete')
print('=' * 60)
print(f'''
📊 Dataset Summary:
   Total samples: {NUM_TRAIN + NUM_VAL + NUM_TEST}
   Train: {NUM_TRAIN} | Val: {NUM_VAL} | Test: {NUM_TEST}

📝 Corpus:
   Arabic prescriptions: {len(MEDICAL_CORPUS["prescriptions_ar"])}
   Arabic lab reports: {len(MEDICAL_CORPUS["lab_reports_ar"])}
   Arabic diagnoses: {len(MEDICAL_CORPUS["diagnoses_ar"])}
   Arabic patient info: {len(MEDICAL_CORPUS["patient_info_ar"])}
   Arabic radiology: {len(MEDICAL_CORPUS["radiology_ar"])}
   English medical: {len(MEDICAL_CORPUS["medical_en"])}
   Medical numbers: {len(MEDICAL_CORPUS["numbers_medical"])}

🔤 Fonts: {len(valid_fonts)}

🎨 Augmentations (14 types):
   - Gaussian noise, Salt & pepper noise
   - Motion blur, Gaussian blur
   - Contrast & brightness adjustment
   - Rotation, Perspective distortion
   - Ink bleed, Paper texture
   - JPEG compression, Shadow
   - Random erasing, Elastic distortion

📦 Export Formats:
   - LMDB (fast training)
   - HuggingFace Dataset
   - JSONL
   - TSV (labels.txt)

🧠 Model Ready:
   - TrOCR (microsoft/trocr-large-handwritten)
   - LoRA fine-tuning ready
   - Arabic vocabulary extended

📂 Location: {OUTPUT_DIR}

🚀 Next Steps:
   1. Run LoRA training: train_trocr_lora.py
   2. Export to HuggingFace Hub
   3. Integrate with OmniMedical Suite OCR pipeline
   4. Run CER/WER evaluation
''')
print('=' * 60)